In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv(r"C:\Users\Tanuja\Downloads\breast_cancer_wisconsin.csv")

df.columns = df.columns.astype(str).str.strip()

print("Dataset Columns:", df.columns.tolist())

target_names = ["diagnosis", "class", "target", "label", "outcome", "result"]

target_col = None

for col in df.columns:
    if col.lower().replace(" ", "").replace("_", "") in target_names:
        target_col = col
        break

if target_col is None:
    last_col = df.columns[-1]
    if 2 <= df[last_col].nunique() <= 10:
        target_col = last_col
    else:
        raise ValueError("Target column not identified. Check the dataset columns.")

print("\nTarget Column:", target_col)

df = df.dropna(subset=[target_col])

y = LabelEncoder().fit_transform(df[target_col].astype(str))

X = df.drop(columns=[target_col])

id_cols = [col for col in X.columns
           if col.lower().replace("_", "").replace(" ", "") in ["id", "index", "samplecodenumber"]
           or col.lower().startswith("unnamed")]

X = X.drop(columns=id_cols, errors="ignore")
X = X.select_dtypes(include="number")
X = X.dropna(axis=1, how="all")
X = X.replace([np.inf, -np.inf], np.nan)

print("\nDataset Shape:", df.shape)
print("Number of Features:", X.shape[1])

if X.shape[1] == 0 or len(np.unique(y)) < 2:
    raise ValueError("Valid numerical features or target classes not found.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

    print("\n", name)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

results_df = pd.DataFrame(results)

print("\nPerformance Comparison:")
print(results_df.round(4).to_string(index=False))

Dataset Columns: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension', 'target']

Target Column: target

Dataset Shape: (569, 31)
Number of Features: 30

 Logistic Regression
Confusion Matrix:
[[41  1]
 [ 1 71]]

 Random Forest
Confusion Matrix:
[[39  3]
 [ 2 70]]

Performance Comparison:
              Model  Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.9825     0.9825  0.9825    0.9825
      Random Forest    0.9561     0.9561  0.9561    0.9560


In [ ]:
%pip install pandas numpy scikit-learn
# PRACTICAL 10: ENSEMBLE LEARNING

## 1. Aim

To analyze the performance of ensemble learning techniques, such as Random Forest, on a suitable open-access dataset in comparison with a baseline machine learning model.

## 2. Dataset Used

**Dataset Name:** Breast Cancer Wisconsin Dataset  
**File Name:** `breast_cancer_wisconsin.csv`  
**File Path:** `C:\Users\Tanuja\Downloads\breast_cancer_wisconsin.csv`

The Breast Cancer Wisconsin dataset contains information about breast cancer samples. It includes numerical measurements that can be used to classify samples into different categories.

The dataset generally contains two classes:

- **Benign:** Non-cancerous tumour.
- **Malignant:** Cancerous tumour.

The program uses numerical features as input and the diagnosis or class column as the target.

**Note:** The exact target column name is detected by the program. Check the printed `Target Column` and confirm that it represents the actual class or diagnosis label in your CSV file.

## 3. Theory

### What is Machine Learning?

Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed for every task.

Machine Learning algorithms learn from training data and use the learned patterns to predict results on new data.

### What is Ensemble Learning?

Ensemble Learning is a machine learning technique that combines the predictions of multiple models to produce a final prediction.

Instead of depending on only one model, ensemble learning uses multiple models to improve prediction performance, stability and generalization.

The main idea is that combining multiple models can produce better results than using a single model, depending on the dataset and algorithms used.

### Types of Ensemble Learning

**1. Bagging (Bootstrap Aggregating)**

Bagging is an ensemble technique in which multiple models are trained independently on different randomly sampled subsets of the training dataset.

Their predictions are combined to produce the final output.

For classification, the final prediction is generally decided using majority voting.

Example: Random Forest.

**2. Boosting**

Boosting is an ensemble technique in which models are trained sequentially. Each new model focuses on correcting errors made by previous models.

The predictions of multiple models are combined to produce the final result.

Examples:

- AdaBoost
- Gradient Boosting
- XGBoost

**3. Stacking**

Stacking combines predictions from multiple base models using another model called a meta-model.

The meta-model learns how to combine the predictions of the base models.

### What is Random Forest?

Random Forest is an ensemble learning algorithm used for classification and regression tasks.

It creates multiple decision trees and combines their predictions to produce the final result.

For classification, Random Forest generally uses majority voting among the individual decision trees.

For example, if 100 decision trees predict whether a sample is benign or malignant, the class receiving the majority of votes becomes the final prediction.

### Working of Random Forest

1. Select multiple random samples from the training dataset using bootstrap sampling.
2. Train a decision tree on each sample.
3. At each split in a tree, consider a random subset of available features.
4. Select the best feature from the considered subset for splitting.
5. Repeat the process to construct multiple decision trees.
6. Each decision tree predicts a class for the input data.
7. Combine the predictions using majority voting.
8. Return the class with the highest number of votes as the final prediction.

### What is Logistic Regression?

Logistic Regression is a supervised machine learning algorithm mainly used for classification problems.

It predicts the probability that an input belongs to a particular class.

For binary classification, it uses the sigmoid function to convert a linear combination of input features into a probability between 0 and 1.

The sigmoid function is:

P(y = 1) = 1 / (1 + e⁻ᶻ)

Where:

z = b₀ + b₁x₁ + b₂x₂ + ... + bₙxₙ

- b₀ = Intercept.
- b₁, b₂, ..., bₙ = Model coefficients.
- x₁, x₂, ..., xₙ = Input features.
- P(y = 1) = Probability of belonging to class 1.

In this practical, Logistic Regression is used as the baseline model for comparison with Random Forest.

### What is a Baseline Model?

A baseline model is a simple reference model used to evaluate the performance of a more advanced model.

In this practical:

- Logistic Regression is the baseline model.
- Random Forest is the ensemble learning model.

Both models are trained and tested using the same dataset split so that their performance can be compared fairly.

### Difference Between Logistic Regression and Random Forest

| Logistic Regression | Random Forest |
|---|---|
| Baseline classification algorithm | Ensemble learning algorithm |
| Uses a mathematical linear relationship with a logistic link | Combines multiple decision trees |
| Predicts class probabilities | Uses voting among trees for classification |
| Generally requires feature scaling | Generally does not require feature scaling |
| A single model | Multiple decision trees |
| Can be limited when class boundaries are highly nonlinear | Can learn complex nonlinear patterns |

## 4. Performance Evaluation Metrics

To compare the two models, four evaluation metrics are used.

### 1. Accuracy

Accuracy measures the proportion of correctly predicted samples out of all tested samples.

Accuracy = (TP + TN) / (TP + TN + FP + FN)

### 2. Precision

Precision measures how many samples predicted as positive are actually positive.

Precision = TP / (TP + FP)

### 3. Recall

Recall measures how many actual positive samples are correctly identified by the model.

Recall = TP / (TP + FN)

### 4. F1-Score

F1-score is the harmonic mean of Precision and Recall.

F1-Score = 2 × (Precision × Recall) / (Precision + Recall)

Where:

- TP = True Positive.
- TN = True Negative.
- FP = False Positive.
- FN = False Negative.

### What is a Confusion Matrix?

A confusion matrix is a table used to evaluate a classification model by comparing actual labels with predicted labels.

For binary classification, it contains four values:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| Actual Positive | True Positive | False Negative |
| Actual Negative | False Positive | True Negative |

It helps identify the types of classification errors made by a model.

### Why are these metrics important?

Accuracy alone may not explain the complete performance of a classification model, especially when the classes are imbalanced.

Precision, Recall and F1-score provide additional information about the model's classification performance.

For medical classification problems, Recall is especially important because false negatives may mean that actual positive cases are missed.

## 5. Algorithm

1. Import the required Python libraries.
2. Load the Breast Cancer Wisconsin dataset.
3. Display the dataset column names.
4. Identify the target column containing the class or diagnosis labels.
5. Remove records with missing target values.
6. Convert categorical target labels into numerical values using LabelEncoder.
7. Separate the input features and target variable.
8. Remove identification columns and non-numerical features.
9. Remove columns containing only missing values.
10. Replace infinite values with missing values.
11. Divide the dataset into training and testing sets.
12. Create a Logistic Regression pipeline with missing-value imputation and feature scaling.
13. Create a Random Forest pipeline with missing-value imputation.
14. Train both models using the training dataset.
15. Predict the classes for the testing dataset.
16. Calculate Accuracy, Precision, Recall and F1-score.
17. Display the confusion matrix for each model.
18. Compare the performance of both models.

## 6. Python Program

### Installation command

Run this command only if the required libraries are not installed.

```python
%pip install pandas numpy scikit-learn
```

### Main Program

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv(r"C:\Users\Tanuja\Downloads\breast_cancer_wisconsin.csv")

df.columns = df.columns.astype(str).str.strip()

print("Dataset Columns:", df.columns.tolist())

target_names = ["diagnosis", "class", "target", "label", "outcome", "result"]

target_col = None

for col in df.columns:
    if col.lower().replace(" ", "").replace("_", "") in target_names:
        target_col = col
        break

if target_col is None:
    last_col = df.columns[-1]
    if 2 <= df[last_col].nunique() <= 10:
        target_col = last_col
    else:
        raise ValueError("Target column not identified. Check the dataset columns.")

print("\nTarget Column:", target_col)

df = df.dropna(subset=[target_col])

y = LabelEncoder().fit_transform(df[target_col].astype(str))

X = df.drop(columns=[target_col])

id_cols = [col for col in X.columns
           if col.lower().replace("_", "").replace(" ", "") in ["id", "index", "samplecodenumber"]
           or col.lower().startswith("unnamed")]

X = X.drop(columns=id_cols, errors="ignore")
X = X.select_dtypes(include="number")
X = X.dropna(axis=1, how="all")
X = X.replace([np.inf, -np.inf], np.nan)

print("\nDataset Shape:", df.shape)
print("Number of Features:", X.shape[1])

if X.shape[1] == 0 or len(np.unique(y)) < 2:
    raise ValueError("Valid numerical features or target classes not found.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

    print("\n", name)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

results_df = pd.DataFrame(results)

print("\nPerformance Comparison:")
print(results_df.round(4).to_string(index=False))
```

## 7. Line-by-Line Explanation of the Program

### A. Importing Libraries

**Line 1:**
```python
import pandas as pd
```
Imports the Pandas library with the alias `pd`. It is used to load and manipulate the dataset.

**Line 2:**
```python
import numpy as np
```
Imports NumPy with the alias `np`. It is used for numerical operations and handling infinite values.

**Line 3:**
```python
from sklearn.model_selection import train_test_split
```
Imports the function used to divide the dataset into training and testing sets.

**Line 4:**
```python
from sklearn.preprocessing import StandardScaler, LabelEncoder
```
Imports two preprocessing techniques.

- StandardScaler standardizes numerical input features.
- LabelEncoder converts categorical target labels into numerical values.

**Line 5:**
```python
from sklearn.impute import SimpleImputer
```
Imports SimpleImputer, which is used to replace missing input values.

**Line 6:**
```python
from sklearn.pipeline import Pipeline
```
Imports Pipeline, which combines preprocessing steps and a machine learning model into one sequence.

**Line 7:**
```python
from sklearn.linear_model import LogisticRegression
```
Imports Logistic Regression, which is used as the baseline classification model.

**Line 8:**
```python
from sklearn.ensemble import RandomForestClassifier
```
Imports RandomForestClassifier, which is used as the ensemble learning model.

**Line 9:**
```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
```
Imports evaluation functions used to measure and compare the performance of both models.

### B. Loading the Dataset

**Line 10:**
```python
df = pd.read_csv(r"C:\Users\Tanuja\Downloads\breast_cancer_wisconsin.csv")
```
Loads the Breast Cancer Wisconsin CSV file into a Pandas DataFrame named `df`.

The letter `r` creates a raw string so that Windows backslashes are handled correctly.

**Line 11:**
```python
df.columns = df.columns.astype(str).str.strip()
```
Converts column names into strings and removes unnecessary spaces from their beginning and end.

**Line 12:**
```python
print("Dataset Columns:", df.columns.tolist())
```
Displays all column names in the dataset as a Python list. This helps verify the available attributes.

### C. Identifying the Target Column

**Line 13:**
```python
target_names = ["diagnosis", "class", "target", "label", "outcome", "result"]
```
Creates a list containing common names used for the target column.

**Line 14:**
```python
target_col = None
```
Creates a variable to store the detected target column name. Initially, it is set to `None`.

**Lines 15–18:**
```python
for col in df.columns:
    if col.lower().replace(" ", "").replace("_", "") in target_names:
        target_col = col
        break
```
Checks each column name to find a match with the target names.

- `df.columns` accesses all column names.
- `lower()` converts the column name into lowercase.
- `replace(" ", "")` removes spaces.
- `replace("_", "")` removes underscores.
- `in target_names` checks whether the cleaned column name matches one of the recognized names.
- `target_col = col` stores the matching column.
- `break` stops the loop after finding a match.

**Lines 19–24:**
```python
if target_col is None:
    last_col = df.columns[-1]
    if 2 <= df[last_col].nunique() <= 10:
        target_col = last_col
    else:
        raise ValueError("Target column not identified. Check the dataset columns.")
```
Checks whether a target column was identified.

If no common target name is found, the program checks the last column. If that column contains between 2 and 10 unique values, it is assumed to be the target.

Otherwise, the program raises an error instead of silently selecting an unsuitable column.

**Line 25:**
```python
print("\nTarget Column:", target_col)
```
Displays the detected target column.

### D. Preparing the Target Variable

**Line 26:**
```python
df = df.dropna(subset=[target_col])
```
Removes records where the target column has missing values because a model cannot learn the correct class for those records.

**Line 27:**
```python
y = LabelEncoder().fit_transform(df[target_col].astype(str))
```
Converts the target labels into numerical values.

- `astype(str)` converts target values into strings.
- `LabelEncoder()` creates a label encoder.
- `fit_transform()` learns the available categories and converts them into numbers.

For example, two categories such as Benign and Malignant can be encoded as 0 and 1.

### E. Preparing Input Features

**Line 28:**
```python
X = df.drop(columns=[target_col])
```
Removes the target column from the dataset. The remaining columns are stored in `X` as input features.

**Lines 29–31:**
```python
id_cols = [col for col in X.columns
           if col.lower().replace("_", "").replace(" ", "") in ["id", "index", "samplecodenumber"]
           or col.lower().startswith("unnamed")]
```
Creates a list of identification columns that should not be used for training.

- `X.columns` accesses feature names.
- The cleaned column name is checked against known ID and index names.
- `startswith("unnamed")` identifies unnecessary unnamed index columns.

**Line 32:**
```python
X = X.drop(columns=id_cols, errors="ignore")
```
Removes the identified ID and index columns. `errors="ignore"` prevents an error if none of the specified columns exist.

**Line 33:**
```python
X = X.select_dtypes(include="number")
```
Selects only numerical columns because the machine learning models require numerical input features.

**Line 34:**
```python
X = X.dropna(axis=1, how="all")
```
Removes columns containing only missing values.

**Line 35:**
```python
X = X.replace([np.inf, -np.inf], np.nan)
```
Replaces positive and negative infinite values with missing values so that they can be handled by SimpleImputer.

**Lines 36–37:**
```python
print("\nDataset Shape:", df.shape)
print("Number of Features:", X.shape[1])
```
Displays the dataset dimensions and the number of numerical input features selected for training.

**Lines 38–39:**
```python
if X.shape[1] == 0 or len(np.unique(y)) < 2:
    raise ValueError("Valid numerical features or target classes not found.")
```
Checks whether the dataset has at least one input feature and at least two target classes.

If these conditions are not satisfied, the program displays an error.

### F. Splitting the Dataset

**Lines 40–42:**
```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```
Divides the dataset into training and testing sets.

- `X` represents input features.
- `y` represents target labels.
- `test_size=0.2` allocates 20% of the records for testing and 80% for training.
- `random_state=42` makes the split reproducible.
- `stratify=y` maintains approximately the same class proportions in the training and testing sets.

### G. Creating the Models

**Lines 43–47: Logistic Regression**

```python
"Logistic Regression": Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
]),
```

Creates the Logistic Regression baseline model using a Pipeline.

- `SimpleImputer(strategy="median")` replaces missing numerical values using the median of each feature.
- `StandardScaler()` standardizes the input features.
- `LogisticRegression(max_iter=2000)` creates the Logistic Regression model with a maximum of 2000 iterations.

The pipeline applies these operations in the correct order.

**Lines 49–52: Random Forest**

```python
"Random Forest": Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])
```

Creates the Random Forest ensemble model using a Pipeline.

- `SimpleImputer(strategy="median")` handles missing numerical values.
- `RandomForestClassifier()` creates the Random Forest classification model.
- `n_estimators=100` specifies that the model will use 100 decision trees.
- `random_state=42` makes its random operations reproducible.

Unlike Logistic Regression, Random Forest does not require feature scaling for its usual decision-tree splits.

### H. Training and Testing Models

**Line 54:**
```python
results = []
```
Creates an empty list to store the performance results of both models.

**Line 55:**
```python
for name, model in models.items():
```
Starts a loop that accesses each model from the dictionary.

- `name` stores the model name.
- `model` stores the corresponding Pipeline.

**Line 56:**
```python
model.fit(X_train, y_train)
```
Trains the selected model using the training features and target labels.

**Line 57:**
```python
y_pred = model.predict(X_test)
```
Uses the trained model to predict target classes for the testing data.

**Lines 59–66:**
```python
results.append({
    "Model": name,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
})
```

Calculates and stores the performance metrics for the current model.

- `accuracy_score()` calculates the proportion of correct predictions.
- `precision_score()` calculates weighted precision.
- `recall_score()` calculates weighted recall.
- `f1_score()` calculates weighted F1-score.
- `average="weighted"` calculates metrics for each class and combines them according to the number of actual samples in each class.
- `zero_division=0` returns zero if a metric has a zero denominator instead of raising a warning.

**Lines 68–70:**
```python
print("\n", name)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
```
Displays the current model name and its confusion matrix.

The confusion matrix compares actual class labels with predicted class labels.

### I. Performance Comparison

**Line 71:**
```python
results_df = pd.DataFrame(results)
```
Converts the list of model evaluation results into a Pandas DataFrame.

**Line 72:**
```python
print("\nPerformance Comparison:")
```
Displays the heading for the comparison table.

**Line 73:**
```python
print(results_df.round(4).to_string(index=False))
```
Displays the performance comparison of Logistic Regression and Random Forest.

- `round(4)` rounds the evaluation metrics to four decimal places.
- `to_string(index=False)` displays the table without row index numbers.

## 8. Expected Output

The program displays:

1. Dataset column names.
2. Detected target column.
3. Dataset shape and number of selected features.
4. Confusion matrix for Logistic Regression.
5. Confusion matrix for Random Forest.
6. Performance comparison of both models using Accuracy, Precision, Recall and F1-score.

**Note:** The exact performance scores depend on the actual CSV dataset and its target labels. Record the values obtained from your own execution rather than assuming that Random Forest will always perform better.

## 9. Important Viva Questions and Answers

**Q1. What is Ensemble Learning?**

Ensemble Learning is a machine learning technique that combines predictions from multiple models to produce a final prediction.

**Q2. What is Random Forest?**

Random Forest is an ensemble learning algorithm that combines multiple decision trees to perform classification or regression.

**Q3. What is a baseline model?**

A baseline model is a reference model used to compare the performance of other machine learning models.

**Q4. Which baseline model is used in this practical?**

Logistic Regression.

**Q5. Which ensemble model is used in this practical?**

Random Forest Classifier.

**Q6. How many decision trees are used in this practical?**

100 decision trees, specified using `n_estimators=100`.

**Q7. What is Bagging?**

Bagging is an ensemble technique that trains multiple models independently on bootstrap samples and combines their predictions.

**Q8. What is Boosting?**

Boosting is an ensemble technique that trains models sequentially, with each new model focusing on correcting errors made by previous models.

**Q9. What is Logistic Regression?**

Logistic Regression is a supervised classification algorithm that predicts the probability of a sample belonging to a particular class.

**Q10. What is a confusion matrix?**

A confusion matrix is a table that compares actual and predicted class labels to evaluate classification performance.

**Q11. What is Accuracy?**

Accuracy is the proportion of correctly predicted samples out of all tested samples.

**Q12. What is Precision?**

Precision measures how many samples predicted as positive are actually positive.

**Q13. What is Recall?**

Recall measures how many actual positive samples are correctly identified.

**Q14. What is F1-score?**

F1-score is the harmonic mean of Precision and Recall.

**Q15. Why is feature scaling used for Logistic Regression?**

Feature scaling brings input features to a comparable range and helps numerical optimization during model training.

**Q16. Is feature scaling necessary for Random Forest?**

Generally, no. Decision trees use feature thresholds and are usually not sensitive to differences in feature scale.

**Q17. Why is train_test_split used?**

It divides the dataset into training and testing portions to evaluate how well a model predicts unseen data.

## 10. Advanced External Viva Questions

**Q1. Why can ensemble learning perform better than a single model?**

Combining multiple models can reduce prediction variance and improve robustness. However, improvement depends on the quality and diversity of the individual models.

**Q2. What is the difference between Bagging and Boosting?**

| Bagging | Boosting |
|---|---|
| Models are generally trained independently | Models are trained sequentially |
| Uses bootstrap samples | New models focus on previous errors |
| Mainly helps reduce variance | Can reduce bias and improve prediction |
| Example: Random Forest | Examples: AdaBoost and Gradient Boosting |

**Q3. Why is Random Forest less prone to overfitting than a single decision tree?**

It combines predictions from multiple trees trained using different samples and feature subsets. This generally reduces the variance of predictions compared with relying on one tree.

**Q4. What is bootstrap sampling?**

Bootstrap sampling creates random samples from a dataset with replacement. Some records may appear more than once, while some may not appear in a particular sample.

**Q5. What is majority voting?**

Majority voting selects the class predicted by the largest number of individual models.

**Q6. What is the role of random feature selection in Random Forest?**

It introduces diversity among trees by allowing each split to consider only a subset of available features.

**Q7. What happens if the number of trees is increased?**

Increasing the number of trees may improve prediction stability, but it also increases computational time and memory usage. Performance does not necessarily improve indefinitely.

**Q8. Why is Recall important in breast cancer classification?**

Recall is important because a false negative may mean that a malignant case is incorrectly predicted as benign. High recall for the malignant class helps reduce missed positive cases.

**Q9. Can Random Forest be used for regression?**

Yes. RandomForestRegressor is used for regression problems, while RandomForestClassifier is used for classification.

**Q10. Does Random Forest always outperform Logistic Regression?**

No. The performance depends on the dataset, features, model parameters and evaluation metric. The actual results must be compared.

## 11. Dataset-Specific Viva Questions

**Q1. Why is the Breast Cancer Wisconsin dataset used?**

It contains numerical features and class labels that make it suitable for demonstrating supervised classification and model comparison.

**Q2. What are the two main classes in the dataset?**

Benign and Malignant.

**Q3. What is a benign tumour?**

A benign tumour is non-cancerous.

**Q4. What is a malignant tumour?**

A malignant tumour is cancerous and may invade surrounding tissues.

**Q5. What is the target variable?**

The target variable represents the diagnosis or class label that the model attempts to predict.

**Q6. Why are ID columns removed?**

ID columns are used for record identification and generally do not provide meaningful predictive information.

**Q7. Why is LabelEncoder used?**

It converts categorical class labels into numerical values so that machine learning algorithms can process them.

**Q8. Why is the dataset divided into training and testing sets?**

The training set is used to train the model, while the testing set evaluates its performance on unseen records.

## 12. Important Points to Remember for Viva

- Practical: Ensemble Learning.
- Dataset: Breast Cancer Wisconsin.
- Problem type: Classification.
- Baseline model: Logistic Regression.
- Ensemble model: Random Forest Classifier.
- Number of trees: 100.
- Ensemble technique used: Bagging.
- Train-test split: 80:20.
- Missing-value handling: Median imputation.
- Feature scaling: StandardScaler for Logistic Regression.
- Target encoding: LabelEncoder.
- Evaluation metrics: Accuracy, Precision, Recall and F1-score.
- Additional evaluation: Confusion Matrix.
- Random Forest combines predictions from multiple decision trees.
- Logistic Regression provides the baseline for comparison.